# 02 · Exploración de los datos de dengue en Cali (2020-2025)

**Proyecto:** Alerta Dengue Cali — Secretaría de Salud Pública Distrital de Santiago de Cali
**Autores:** Samuel Acevedo Mambuscay y Luis Carlos Tez — Universidad Libre, seccional Cali

En este cuaderno revisamos qué llegó de cada fuente (Bronze), cómo quedó después de la
limpieza (Silver) y qué muestran las tablas de consumo (Gold). Las preguntas que guían la
exploración son las del documento del proyecto:

1. ¿Cuántos casos y qué incidencia tuvo Cali cada semana y cada año?
2. ¿En qué semanas la ciudad estuvo por encima de su canal endémico (semanas inusuales)?
3. ¿Hay un patrón estacional y qué tanto se parece una semana a la anterior?
4. ¿Qué grupos de población concentran más casos y más severidad?
5. ¿Cuánto tardan los pacientes en consultar y se relaciona con la gravedad?
6. ¿Qué comunas tienen mayor incidencia?
7. ¿Hay registros inconsistentes o lecturas climáticas anómalas?
8. ¿El clima de las semanas anteriores anticipa el aumento de casos?

El cuaderno **no** transforma datos: lee las capas que genera `make pipeline` y llama a las
funciones de `flows/`. Las conclusiones quedan en `reports/conclusiones_eda.md`.

In [ ]:
import importlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from flows.common.config import settings
from flows.common.epidemiologia import GRUPOS_EDAD, ZONAS_CANAL
from flows.common.paths import (
    BRONZE,
    BRONZE_METADATA_NAME,
    BRONZE_QUALITY_NAME,
    FIGURES,
    GOLD,
    SILVER,
    SILVER_CASOS,
    SILVER_CLEANING_NAME,
    SILVER_CLIMA,
    SILVER_POBLACION,
    SILVER_SERIE,
)

agg = importlib.import_module("flows.03_aggregation")   # nombre con dígito: se usa importlib

sns.set_theme(style="whitegrid", context="notebook")
FIGURES.mkdir(parents=True, exist_ok=True)
COLORES_ZONA = {"Éxito": "#2e7d32", "Seguridad": "#f9a825", "Alerta": "#ef6c00",
                "Brote": "#c62828"}


def guardar(fig, nombre):
    """Guarda la figura en reports/figuras para usarla en los informes."""
    fig.savefig(FIGURES / f"{nombre}.png", dpi=150, bbox_inches="tight")

## 1. Bronze: lo que llegó de las fuentes

Cada archivo descargado de los portales se guardó sin modificar sus valores, con su huella
SHA-256. De los microdatos nacionales del INS solo se copiaron las filas con procedencia
Cali (departamento 76, municipio 001), y del Excel de proyecciones municipales del DANE,
solo las filas de Cali (76001). Las series completas del IDEAM y las proyecciones del DANE
se obtienen con `make descargar` desde las API de datos abiertos.

In [ ]:
metadatos = json.loads((BRONZE / BRONZE_METADATA_NAME).read_text(encoding="utf-8"))
calidad = json.loads((BRONZE / BRONZE_QUALITY_NAME).read_text(encoding="utf-8"))
inventario = pd.DataFrame([
    {"fuente": nombre, "origen": info["origin"], "archivo_bronze": info["bronze_file"],
     "filas_origen": info["rows_source"], "filas_bronze": info["rows_bronze"],
     "columnas": calidad[nombre]["n_cols"], "filas_vacias": calidad[nombre]["empty_rows"],
     "sha256": info["sha256_bronze"][:12] + "…"}
    for nombre, info in metadatos["files"].items()
])
print(f"{metadatos['n_files']} archivos, {metadatos['rows_bronze_total']:,} filas en Bronze")
inventario

Dos archivos del portal del Distrito vienen inflados con filas vacías: el de 2022 (51.088
vacías de 53.792) y el de mortalidad 2023 (53.790 vacías; solo una fila tiene un caso).
Eso no es un error de lectura: el archivo publicado es así, y la limpieza lo descarta.

## 2. Silver: qué hizo la limpieza

Silver une las cuatro fuentes de casos en un esquema común (un caso por fila). La serie
semanal usa la **semana epidemiológica de inicio de síntomas** para todos los años, que es
el criterio con el que el INS publica la variable `SEMANA` de los microdatos.

In [ ]:
limpieza = json.loads((SILVER / SILVER_CLEANING_NAME).read_text(encoding="utf-8"))
casos = pd.read_parquet(SILVER / SILVER_CASOS)
serie = pd.read_parquet(SILVER / SILVER_SERIE)
print(json.dumps({k: limpieza[k] for k in ("casos_con_datos", "duplicados_eliminados",
                                          "casos_en_silver", "fallecidos_integrados")},
                 indent=1))
pd.crosstab(casos["anio_epi"], casos["fuente"], margins=True, margins_name="Total")

**Completitud de las variables clave por fuente.** Los microdatos del INS no traen
síntomas ni clasificación final; por eso el modelo de riesgo solo puede usar los casos del
Distrito.

In [ ]:
clave = ["edad", "sexo", "estrato", "comuna", "afiliacion", "fecha_consulta", "fiebre",
         "severo", "hospitalizado"]
(casos.groupby("fuente")[clave].agg(lambda s: s.notna().mean() * 100).T
 .astype(float).rename(columns=lambda c: f"% con dato ({c})").round(1))

**Registros inconsistentes (detección de anomalías en los datos).** No se borran: quedan
marcados en Silver para que cada análisis decida qué hacer con ellos.

In [ ]:
pd.read_parquet(GOLD / "calidad_datos.parquet").round(2)

## 3. Serie semanal e incidencia anual

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(serie["fecha_inicio"], serie["casos"], color="#37474f", lw=1.3, label="Casos observados")
imputadas = serie[serie["imputada"]]
ax.scatter(imputadas["fecha_inicio"], imputadas["casos_modelo"], color="#ef6c00", s=18,
           zorder=3, label=f"Semanas sin datos o parciales, interpoladas ({len(imputadas)})")
for anio, fuente in serie.assign(f=np.where(serie["anio_epi"].isin([2022, 2023]),
                                            "SSPD Cali", "INS")).groupby("anio_epi")["f"].first(
                                            ).items():
    inicio = serie.loc[serie["anio_epi"] == anio, "fecha_inicio"].min()
    ax.text(inicio, ax.get_ylim()[1] * 0.93, f"{anio}\n{fuente}", fontsize=8, color="#546e7a")
ax.set(title="Casos semanales de dengue en Cali por semana de inicio de síntomas",
       xlabel="", ylabel="Casos por semana")
ax.legend(loc="upper left", bbox_to_anchor=(0.0, 0.8))
guardar(fig, "fig01_serie_semanal")

**Población.** Desde 2018 los totales de Cali son las proyecciones oficiales del DANE (censo
2018, actualización de 2025). La población de cada comuna es el total de la cabecera del
DANE repartido según la participación de la comuna en la proyección de Planeación
Distrital (la de 2020 para los años siguientes).

In [ ]:
poblacion = pd.read_parquet(SILVER / SILVER_POBLACION)
(poblacion[(poblacion["tipo"] == "Ciudad") & poblacion["anio"].between(2016, 2025)]
 [["anio", "poblacion", "fuente_poblacion"]])

In [ ]:
anual = pd.read_parquet(GOLD / "incidencia_anual.parquet")
anual[["anio_epi", "casos", "tasa_100k", "semanas_con_datos", "anio_completo",
       "pct_hospitalizados", "pct_menores_15", "edad_mediana", "fuente"]].round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
colores = np.where(anual["anio_completo"], "#1565c0", "#90a4ae")
barras = ax.bar(anual["anio_epi"].astype(str), anual["tasa_100k"], color=colores)
ax.bar_label(barras, labels=[f"{t:,.0f}".replace(",", ".") for t in anual["tasa_100k"]],
             fontsize=9)
ax.set(title="Incidencia anual de dengue (casos por 100.000 habitantes)",
       ylabel="Casos por 100.000", xlabel="Año (gris: archivo cortado antes de fin de año)")
guardar(fig, "fig02_incidencia_anual")

2024 fue el año de mayor transmisión del periodo: 35.604 casos y 1.561,5 por 100.000
habitantes, 2,7 veces la incidencia de 2020. 2022 y 2023 están subestimados porque el
portal del Distrito publicó los archivos cortados en diciembre (3 y 4 semanas sin datos).

## 4. Canal endémico y semanas inusuales

El canal se calcula con el método de cuartiles (Bortman, 1999) usando como años base los
de baja transmisión del periodo, y una ventana de ±2 semanas para compensar que solo hay
tres años base.

In [ ]:
canal = pd.read_parquet(GOLD / "canal_endemico.parquet")
situacion = pd.read_parquet(GOLD / "situacion_semanal.parquet")
print("Años base:", canal["anios_base"].iloc[0], "| ventana ±", settings.canal_ventana)
zonas = (situacion[situacion["cobertura"] == "completa"]
         .pivot_table(index="anio_epi", columns="zona", values="semana_epi", aggfunc="count",
                      fill_value=0)
         .reindex(columns=ZONAS_CANAL, fill_value=0))
zonas

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
for ax, anio in zip(ejes, [2020, 2024, 2025], strict=True):
    datos = situacion[situacion["anio_epi"] == anio]
    s = datos["semana_epi"]
    ax.fill_between(s, 0, datos["q1"], color=COLORES_ZONA["Éxito"], alpha=0.25, label="Éxito")
    ax.fill_between(s, datos["q1"], datos["mediana"], color=COLORES_ZONA["Seguridad"],
                    alpha=0.25, label="Seguridad")
    ax.fill_between(s, datos["mediana"], datos["q3"], color=COLORES_ZONA["Alerta"], alpha=0.25,
                    label="Alerta")
    ax.plot(s, datos["q3"], color=COLORES_ZONA["Brote"], ls="--", lw=1, label="Q3")
    ax.plot(s, datos["casos"], color="black", lw=1.8, label="Casos")
    ax.set(title=f"{anio}: {int(datos['supera_q3'].sum())} semanas sobre Q3",
           xlabel="Semana epidemiológica")
ejes[0].set_ylabel("Casos por semana")
ejes[0].legend(fontsize=8)
fig.suptitle("Casos frente al canal endémico de Cali", y=1.02)
guardar(fig, "fig03_canal_endemico")

2020 y 2024 pasan casi todo el año en zona de brote; 2025 vuelve al canal desde el primer
trimestre. Que 2021 tenga muchas semanas sobre Q3 a pesar de ser año base muestra la
limitación de un canal con tres años: el primer semestre de 2021 todavía arrastraba la
epidemia de 2020.

## 5. Estacionalidad y autocorrelación

In [ ]:
estacional = pd.read_parquet(GOLD / "estacionalidad.parquet")
acf = pd.read_parquet(GOLD / "autocorrelacion.parquet")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [2, 1]})
ax1.fill_between(estacional["semana_epi"], estacional["indice_min"], estacional["indice_max"],
                 color="#90caf9", alpha=0.5, label="Rango entre años")
ax1.plot(estacional["semana_epi"], estacional["indice_medio"], color="#1565c0", lw=2,
         label="Índice medio")
ax1.axhline(1, color="grey", lw=0.8, ls=":")
ax1.set(title="Índice estacional (1 = semana promedio del año)", xlabel="Semana epidemiológica",
        ylabel="Casos / promedio anual")
ax1.legend()
ax2.bar(acf["rezago_semanas"], acf["autocorrelacion"], color="#455a64")
ax2.set(title="Autocorrelación de la serie", xlabel="Rezago (semanas)", ylim=(0, 1))
guardar(fig, "fig04_estacionalidad_autocorrelacion")
acf.head(4).round(3)

Las semanas 1 a 11 tienen entre 20 % y 50 % más casos que la semana promedio del año y las
últimas semanas del año, menos de la mitad; pero el rango entre años es amplio: la
estacionalidad es más débil que la diferencia entre años epidémicos y no epidémicos. La autocorrelación a una semana es 0,99: el mejor predictor de
la próxima semana es la semana actual, algo que el modelo de pronóstico tiene que superar.

## 6. ¿Quién se enferma y quién se agrava?

In [ ]:
perfil = pd.read_parquet(GOLD / "perfil_demografico.parquet")
edades = casos["edad"]
print(f"Mediana de edad: {edades.median():.0f} años | menores de 15: "
      f"{(edades < 15).mean() * 100:.1f} % (n = {edades.notna().sum():,})")
distrito = casos[casos["fuente"] == "SSPD Cali"]["edad"]
print(f"Casos del Distrito 2022-2023: mediana {distrito.median():.0f} años, menores de 15 "
      f"{(distrito < 15).mean() * 100:.1f} %")
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.barplot(data=perfil, x="grupo_edad", y="casos", hue="sexo", order=GRUPOS_EDAD, ax=ax,
            palette=["#ad1457", "#1565c0"])
ax.set(title="Casos por grupo de edad y sexo (2020-2025)", xlabel="Grupo de edad",
       ylabel="Casos")
guardar(fig, "fig05_casos_edad_sexo")

La severidad (con signos de alarma o dengue grave) solo se puede medir con los casos del
Distrito (2022-2023), que son los que traen la clasificación final. Los intervalos son de
Wilson al 95 %.

In [ ]:
severidad = pd.read_parquet(GOLD / "severidad_por_grupo.parquet")
fig, ejes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
orden = {"grupo_edad": GRUPOS_EDAD}
titulos = {"grupo_edad": "Grupo de edad", "afiliacion": "Régimen de afiliación",
           "estrato": "Estrato"}
for ax, variable in zip(ejes, titulos, strict=True):
    datos = severidad[severidad["variable"] == variable].copy()
    if variable in orden:
        datos["categoria"] = pd.Categorical(datos["categoria"], orden[variable], ordered=True)
    datos = datos.sort_values("categoria")
    error = [datos["pct_severos"] - datos["ic95_inferior"],
             datos["ic95_superior"] - datos["pct_severos"]]
    ax.bar(datos["categoria"].astype(str), datos["pct_severos"], yerr=error, capsize=3,
           color="#c62828", alpha=0.8)
    ax.axhline(casos["severo"].mean() * 100, color="black", ls=":", lw=1, label="Promedio")
    ax.set(title=titulos[variable], xlabel="")
    ax.tick_params(axis="x", rotation=30)
ejes[0].set_ylabel("% con signos de alarma o grave")
ejes[0].legend()
guardar(fig, "fig06_severidad_por_grupo")
severidad.round(1)

In [ ]:
pruebas = json.loads((GOLD / "pruebas_estadisticas.json").read_text(encoding="utf-8"))
pd.DataFrame({k: v for k, v in pruebas.items() if k.startswith("severidad_vs")}).T[
    ["n", "chi2", "dof", "p_value", "cramers_v"]].astype(float).round(4)

Las asociaciones son estadísticamente significativas para edad, afiliación y estrato, pero
**débiles** (V de Cramér entre 0,04 y 0,10): ningún rasgo sociodemográfico separa por sí
solo los casos que se agravan. El sexo no se asocia con la severidad (p = 0,67). Los
mayores de 65 años (51 %) y los no asegurados son los grupos con mayor proporción de casos
severos.

## 7. Oportunidad de consulta y notificación

In [ ]:
display(pd.read_parquet(GOLD / "oportunidad_anual.parquet").round(1))
clasificados = casos[casos["severo"].notna() & ~casos["intervalo_atipico"]].copy()
clasificados["clasificacion"] = np.where(clasificados["severo"].astype(bool),
                                         "Con signos de alarma o grave", "Sin signos de alarma")
fig, ax = plt.subplots(figsize=(8, 3.5))
sns.histplot(data=clasificados, x="dias_sintomas_consulta", hue="clasificacion", discrete=True,
             stat="percent", common_norm=False, multiple="dodge", shrink=0.8, ax=ax,
             palette=["#c62828", "#2e7d32"], hue_order=["Con signos de alarma o grave",
                                                         "Sin signos de alarma"])
ax.set(xlim=(-0.5, 12.5), title="Días entre el inicio de síntomas y la consulta",
       xlabel="Días", ylabel="% de los casos de cada grupo")
guardar(fig, "fig07_oportunidad_consulta")
pruebas["dias_consulta_vs_severidad"]

Los casos severos consultan en mediana un día más tarde (4 frente a 3 días; Mann-Whitney
p < 0,001). Lo interpretamos con cuidado: los signos de alarma suelen aparecer entre el día 3
y el 7 de la enfermedad, así que consultar tarde puede ser consecuencia de esperar a que
aparezcan y no solo una causa de la gravedad. La consulta tardía (más de 3 días) bajó de
59 % en 2020 a 44 % en 2025.

## 8. Comunas

In [ ]:
comunas = pd.read_parquet(GOLD / "indicadores_comunas.parquet")
urbanas = comunas[comunas["zona"] == "Urbana"].sort_values("tasa_100k")
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(urbanas["territorio"], urbanas["tasa_100k"], color="#6a1b9a")
ax.axvline(urbanas["tasa_100k"].median(), color="black", ls=":", lw=1, label="Mediana")
ax.set(title=f"Incidencia por comuna ({urbanas['periodo'].iloc[0]})",
       xlabel="Casos por 100.000 habitantes", ylabel="")
ax.legend()
guardar(fig, "fig08_incidencia_comunas")
urbanas[["comuna", "casos", "tasa_100k", "pct_severos", "pct_hospitalizados", "edad_mediana",
         "estrato_medio", "pct_subsidiado", "hosp_arbovirosis_100k"]].sort_values(
    "tasa_100k", ascending=False).round(1)

In [ ]:
correlacion = urbanas[["tasa_100k", "pct_severos", "pct_hospitalizados", "edad_mediana",
                       "estrato_medio", "pct_subsidiado", "hosp_arbovirosis_100k"]].corr(
    method="spearman")
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(correlacion, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, ax=ax,
            cbar_kws={"label": "Spearman"})
ax.set_title("Correlación entre indicadores de las 22 comunas")
guardar(fig, "fig09_correlacion_comunas")

La incidencia en 2022-2023 va de 177 casos por 100.000 en la comuna 1 a 1.188 en la
comuna 4 (6,7 veces más). Las comunas de estrato más alto tienen pacientes mayores y menos
afiliación subsidiada; esas relaciones son las que aprovecha el agrupamiento del
cuaderno 04. Son correlaciones entre territorios con 22 observaciones: no dicen nada sobre
el riesgo de una persona concreta (falacia ecológica).

## 9. Clima (IDEAM)

Lluvia y temperatura de las estaciones automáticas del IDEAM en Cali, descargadas de la API
de Datos Abiertos para 2020-2025. El clima de la ciudad en cada semana es el promedio de
las tres estaciones urbanas (Universidad del Valle, Base Aérea y Siloé) que recibieron al
menos la mitad de sus lecturas. Antes se revisaron lecturas fuera de rango y repetidas.

In [ ]:
clima = pd.read_parquet(SILVER / SILVER_CLIMA)
print(json.dumps(limpieza["clima"], indent=1))
(serie.assign(con_lluvia=serie["lluvia_mm"].notna(), con_temperatura=serie["temp_media"].notna())
 .groupby("anio_epi")[["con_lluvia", "con_temperatura"]].sum())

**Cambio de sistema de medición.** En la segunda mitad de 2024 el IDEAM empezó a publicar
los sensores con otra descripción ("TEMPERATURA DEL AIRE A 2 m" en lugar de "Temp Aire 2
m"). Con el cambio, la temperatura media registrada en la Universidad del Valle bajó unos
4 °C, y en Siloé y la Base Aérea, entre 1 y 2 °C: un salto que no es climático.

In [ ]:
(clima[clima["codigo_estacion"].isin(["0026055120", "0026085160", "0026085170"])]
 .groupby(["estacion", "sistema"])["temp_media"].agg(["mean", "count"]).round(1))

Por eso, además de la temperatura media, se usa la **anomalía**: la diferencia con la media
de la misma estación y el mismo sistema de medición.

In [ ]:
rezagos = pd.read_parquet(GOLD / "clima_rezagos.parquet")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [3, 2]})
ax1.plot(serie["fecha_inicio"], serie["temp_media"], color="#ef6c00", lw=1.2,
         label="Temperatura media")
ax1.plot(serie["fecha_inicio"], serie["temp_anomalia"] + 24, color="#6a1b9a", lw=1.2,
         label="Anomalía (+24 °C para comparar)")
ax1.set_ylabel("°C")
ax1.legend(loc="upper left", fontsize=8)
eje_casos = ax1.twinx()
eje_casos.fill_between(serie["fecha_inicio"], serie["casos"].astype(float), color="#90a4ae",
                       alpha=0.35)
eje_casos.set_ylabel("Casos por semana (gris)")
eje_casos.grid(False)
ax1.set_title("Temperatura de la ciudad y casos")
estilos = {"temp_media": ("#ef6c00", "--", "Temperatura media"),
           "temp_anomalia": ("#6a1b9a", "-", "Anomalía de temperatura"),
           "lluvia_mm": ("#1565c0", "-", "Lluvia")}
for variable, (color, linea, etiqueta) in estilos.items():
    datos = rezagos[rezagos["variable"] == variable]
    ax2.plot(datos["rezago_semanas"], datos["rho"], marker="o", ls=linea, color=color,
             label=etiqueta)
ax2.axhline(0, color="grey", lw=0.8)
ax2.set(title="Clima de hace k semanas vs. crecimiento\nde los casos en las 4 semanas siguientes",
        xlabel="Rezago k (semanas)", ylabel="Spearman")
ax2.legend(fontsize=8)
fig.tight_layout(w_pad=3)
guardar(fig, "fig10_clima_casos")
rezagos.pivot(index="rezago_semanas", columns="variable", values="rho").round(2)

In [ ]:
{k: v for k, v in pruebas.items() if k.endswith("_vs_casos") or "previa" in k}

La API del IDEAM tiene huecos grandes: hay lluvia en 159 de las 314 semanas y temperatura
en 121 (casi nada en 2021 y 2023). Con lo disponible:

- El clima de la misma semana no se relaciona con el número de casos (rho cercano a 0).
- La temperatura media de las semanas anteriores **parece** anticipar el crecimiento de
  los casos (rho de 0,22 a 0,33), pero es un artefacto: el descenso de la epidemia de 2024
  coincide con el cambio de sistema que bajó la temperatura registrada. Con la anomalía,
  que corrige ese salto, la relación desaparece (rho entre −0,11 y 0,18, ningún p < 0,05).
- La lluvia muestra una relación débil y negativa; la más fuerte es a las 9 semanas
  (rho = −0,24, p = 0,005), entre 13 rezagos probados, así que puede ser casualidad.

Conclusión: con los datos públicos del IDEAM no se encuentra una relación robusta entre el
clima y los casos de Cali. Igual se prueba la temperatura como variable candidata en el
pronóstico (cuaderno 04).

## 10. Resumen

- La serie 2020-2025 tiene 314 semanas; 9 (7 sin datos por el corte de publicación de
  diciembre de 2022 y 2023, y las 2 semanas parciales anteriores) se interpolan solo para
  el modelo y quedan marcadas.
- Dos epidemias: 2020 (13.230 casos) y 2024 (35.604, 1.561,5 por 100.000). 2025 regresó al
  canal endémico.
- La serie es muy persistente (autocorrelación 0,99 a una semana) y la estacionalidad es
  débil frente a la diferencia entre años.
- Severidad: asociaciones significativas pero débiles con edad, afiliación y estrato.
- Incidencia por comuna muy desigual (177 a 1.188 por 100.000 en 2022-2023), con la
  población oficial del DANE.
- Clima: la aparente relación entre la temperatura previa y el crecimiento de los casos
  se debe a un cambio en el sistema de medición del IDEAM; corregido, no hay relación.
  La lluvia muestra una relación débil y negativa a las 9 semanas.

Detalle, cifras con su n y limitaciones: `reports/conclusiones_eda.md`.